# Numerically Estimating Optimal Risk in Proprietary Trading Firms 
The optimal risk management strategy can be precomputed using dynamic programming (value iteration) over a discretised equity grid. The risk that maximises the expected probability of passing the challenge (the value function) is stored in a look-up table.

**Author:** puzzle

**Created:** 2026-03-12

**Modified:** 2026-03-12


In [1]:
import numpy as np
import pandas as pd
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go

In [2]:
from jfmi.plot.utilities import load_plotly_templates

In [3]:
from jfmi.plot.templates import COLOURS

In [4]:
load_plotly_templates()

In [5]:
win_rate = 0.85
risk_to_reward = 0.25
# win_rate = 0.7
# risk_to_reward = 0.5
# win_rate = 0.55
# risk_to_reward = 1.0

win_rate * risk_to_reward + (1 - win_rate) * (-1)  # expected value

0.06249999999999997

In [6]:
pass_level = 10.0 / 100  # fraction profit (from initial balance)
fail_level = 10.0 / 100  # fraction drawdown (from initial balance)

initial_account_equity = 1.0

In [7]:
max_risk = 2.50 / 100
min_risk = 0.50 / 100

risk_step = 0.25 / 100

arr_risk_candidates = np.linspace(
    min_risk,
    max_risk,
    round((max_risk - min_risk) / risk_step) + 1,
)

arr_risk_candidates

array([0.005 , 0.0075, 0.01  , 0.0125, 0.015 , 0.0175, 0.02  , 0.0225,
       0.025 ])

In [8]:
pass_equity = initial_account_equity * (1 + pass_level)
fail_equity = initial_account_equity * (1 - fail_level)

equity_step = 0.001

arr_discretised_equities = np.linspace(
    fail_equity,
    pass_equity,
    round((pass_equity - fail_equity) / equity_step) + 1,
)

arr_discretised_equities

array([0.9  , 0.901, 0.902, 0.903, 0.904, 0.905, 0.906, 0.907, 0.908,
       0.909, 0.91 , 0.911, 0.912, 0.913, 0.914, 0.915, 0.916, 0.917,
       0.918, 0.919, 0.92 , 0.921, 0.922, 0.923, 0.924, 0.925, 0.926,
       0.927, 0.928, 0.929, 0.93 , 0.931, 0.932, 0.933, 0.934, 0.935,
       0.936, 0.937, 0.938, 0.939, 0.94 , 0.941, 0.942, 0.943, 0.944,
       0.945, 0.946, 0.947, 0.948, 0.949, 0.95 , 0.951, 0.952, 0.953,
       0.954, 0.955, 0.956, 0.957, 0.958, 0.959, 0.96 , 0.961, 0.962,
       0.963, 0.964, 0.965, 0.966, 0.967, 0.968, 0.969, 0.97 , 0.971,
       0.972, 0.973, 0.974, 0.975, 0.976, 0.977, 0.978, 0.979, 0.98 ,
       0.981, 0.982, 0.983, 0.984, 0.985, 0.986, 0.987, 0.988, 0.989,
       0.99 , 0.991, 0.992, 0.993, 0.994, 0.995, 0.996, 0.997, 0.998,
       0.999, 1.   , 1.001, 1.002, 1.003, 1.004, 1.005, 1.006, 1.007,
       1.008, 1.009, 1.01 , 1.011, 1.012, 1.013, 1.014, 1.015, 1.016,
       1.017, 1.018, 1.019, 1.02 , 1.021, 1.022, 1.023, 1.024, 1.025,
       1.026, 1.027,

In [9]:
def discretise_equity(equity, fail_equity, equity_step):
    return np.clip(
        round((equity - fail_equity) / equity_step),
        0,
        len(arr_discretised_equities) - 1,
    )

In [10]:
def precompute_risk_management(
    arr_risk_candidates,
    arr_discretised_equities,
    equity_step,
    # pass_equity,
    fail_equity,
    win_rate,
    risk_to_reward,
    threshold=1e-6,
):
    arr_policy = np.zeros(len(arr_discretised_equities))

    # Here, the value function is the probability of passing.
    arr_values = np.zeros(len(arr_discretised_equities))

    arr_values[-1] = 1  # pass state
    arr_values[0] = 0  # fail state

    for _ in range(1000):
        arr_values_new = arr_values.copy()

        for i, equity in enumerate(arr_discretised_equities):
            if i == 0 or i == len(arr_discretised_equities) - 1:
                continue

            best_value = -np.inf
            best_risk = min_risk

            for risk in arr_risk_candidates:
                win_equity = equity * (1 + risk * risk_to_reward)
                lose_equity = equity * (1 - risk)

                win_idx = discretise_equity(win_equity, fail_equity, equity_step)
                loss_idx = discretise_equity(lose_equity, fail_equity, equity_step)

                expected_value = (
                    win_rate * arr_values[win_idx]
                    + (1 - win_rate) * arr_values[loss_idx]
                )

                if expected_value > best_value:
                    best_value = expected_value
                    best_risk = risk

            arr_values_new[i] = best_value
            arr_policy[i] = best_risk

        if np.max(np.abs(arr_values_new - arr_values)) < threshold:
            break

        arr_values = arr_values_new

    return arr_policy

In [11]:
arr_policy = precompute_risk_management(
    arr_risk_candidates,
    arr_discretised_equities,
    equity_step,
    fail_equity,
    win_rate,
    risk_to_reward,
)

In [12]:
df = pd.DataFrame({
    "Equity": arr_discretised_equities,
    "Optimal Risk": arr_policy,
})

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=arr_discretised_equities,
        y=arr_policy,
        mode="lines",
        line=dict(color=COLOURS["purple"][5], width=2),
    )
)

fig.add_vline(
    x=pass_equity,
    line_color=COLOURS["green"][5],
    line_width=2,
    opacity=0.75,
)

fig.add_vline(
    x=fail_equity,
    line_color=COLOURS["red"][5],
    line_width=2,
    opacity=0.75,
)

fig.update_layout(
    title=(
        f"Toy Strategy Optimal Risk "
        f"(Win Rate: {win_rate}, Risk : Reward: {risk_to_reward})"
    ),
    xaxis_title="Risk",
    yaxis_title="Account Equity",
)

fig.show()

This isn't dissimilar from the boundary risk management approximation; you generally see that behaviour replicated here. High expectation value strategies genereally converge on lower risks (as you get to take more trades, reducing variance), whilst negative expectation value strategies generally want to take risks (for the opposite reason), replicating what was visible in the toy strategy scatter plot.

What's different is that the optimal risk management is generally non-constant. Some strategies are better off increasing risk as they draw down, some are better off increasing it when they make progress, and some are best off remaining constant. There can also be wiggles at the edges of these zones, which ensure the trade before the boundary trades land in the right places.

Attempting to correct for the unrealised drawdowns being considered when checking for failure will be difficult. How can you accurately estimate the intra-trade drawdown? Strategies with a high win rate and expected value might suggest market timing ability, meaning the intra-trade drawdown is likely to be low. Strategies with a high win rate but zero expected value do not. The maximum drawdown depth and length percentiles (or perhaps ulcer index) can be extracted from backtests.

Still, changing strategies to something riskier that relies on market timing ability and has a high risk:reward below some threshold seems appropriate. That threshold should be above the point at which the optimal risk begins to ramp up, at perhaps the fail level + ~2%.

### For example:
```python
if lose_equity <= fail_equity:
    lose_value = CORRECTION_FUNCTION(win_rate, risk_to_reward)
else:
    loss_idx = discretise_equity(lose_equity, fail_equity, equity_step)
    loss_value = arr_values[loss_idx]

expected_value = win_rate * arr_values[win_idx] + (1 - win_rate) * loss_value

```
